# TP MCP local + OpenAI

Objectif du TP : comprendre le rôle de MCP avec un exemple minimal.

À la fin, on veut que le schéma mental soit clair :

```text
OpenAI choisit un tool
        ↓
le notebook reçoit le tool_call
        ↓
le notebook appelle le serveur MCP local
        ↓
le serveur MCP exécute le vrai code
        ↓
le résultat revient à OpenAI
        ↓
OpenAI formule la réponse finale
```

Message à retenir :

> MCP ne remplace pas le function calling.  
> MCP standardise l'exposition des tools ; le function calling permet au modèle de demander leur utilisation.

Dans ce notebook, le serveur MCP est lancé automatiquement par le notebook en mode `stdio`.
Vous n'avez donc pas besoin d'ouvrir un terminal séparé pour ce TP.

## 0. Installation

À exécuter une seule fois si l'environnement n'est pas déjà prêt.

In [5]:
"""Installation des dépendances pour le TP.

Cette cellule essaie d'abord d'activer `pip` dans l'environnement courant,
puis installe les bibliothèques nécessaires :
- openai
- mcp[cli]
- python-dotenv

Si vous avez déjà tout installé dans votre environnement, vous pouvez
passer cette cellule.
"""

import sys
import subprocess

try:
    import pip  # type: ignore[unused-import]
except ModuleNotFoundError:
    # Certains environnements n'ont pas `pip` activé par défaut
    # On essaie de l'initialiser proprement.
    import ensurepip

    ensurepip.bootstrap()

packages = ["openai", "mcp[cli]", "python-dotenv"]

print("Installation / mise à jour des paquets pour le TP...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])
print("✔ Dépendances installées. Vous pouvez passer à la suite.")

Installation / mise à jour des paquets pour le TP...
✔ Dépendances installées. Vous pouvez passer à la suite.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


## 1. Imports et configuration

Il faut une clé OpenAI disponible dans l'environnement :

```bash
export OPENAI_API_KEY="sk-..."
```

ou dans un fichier `.env`.

In [6]:
import os
import json
from pprint import pprint
from openai import OpenAI
import os
import sys
import json
from pathlib import Path
from pprint import pprint

from openai import OpenAI

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


if ("OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"].strip()):
    with open("openai_api_key.txt", "r") as f:
        api_key = f.read().strip()
    os.environ["OPENAI_API_KEY"] = api_key
else:
    api_key = os.environ["OPENAI_API_KEY"]


client = OpenAI()

## 2. Créer un petit serveur MCP local

Dans un vrai projet, le serveur MCP serait souvent fourni par un outil externe ou une équipe métier.

Ici, on crée un serveur minimal dans un fichier Python : `mcp_tp_server.py`.

Il expose deux tools :

- `calculate_cart_total`
- `divide`

Important : ces tools ne sont pas définis directement dans OpenAI. Ils sont exposés par un serveur MCP.

In [7]:
SERVER_FILE = Path("mcp_tp_server.py")

server_code = """
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("tp-mcp-demo")

@mcp.tool()
def calculate_cart_total(
    item_prices: list[float],
    discount_percent: float,
    tax_percent: float,
    shipping_fee: float,
    free_shipping_threshold: float,
) -> dict:
    \"""
    Calculate the final price of a shopping cart.

    Rules:
    - Sum item prices.
    - Apply discount.
    - Apply tax after discount.
    - Add shipping unless total with tax before shipping is strictly greater than the free shipping threshold.
    - Round values to two decimals.
    \"""
    subtotal = sum(item_prices)
    discounted_total = subtotal * (1 - discount_percent / 100)
    total_with_tax = discounted_total * (1 + tax_percent / 100)
    shipping_applied = 0 if total_with_tax > free_shipping_threshold else shipping_fee
    final_total = total_with_tax + shipping_applied

    return {
        "subtotal": round(subtotal, 2),
        "discounted_total": round(discounted_total, 2),
        "total_with_tax_before_shipping": round(total_with_tax, 2),
        "shipping_applied": round(shipping_applied, 2),
        "final_total": round(final_total, 2),
    }


@mcp.tool()
def divide(a: float, b: float) -> dict:
    \"""
    Divide a by b. If b is zero, return a structured error instead of crashing.
    \"""
    if b == 0:
        return {
            "error": "division_by_zero",
            "message": "Division by zero is not defined."
        }
    return {"result": a / b}


if __name__ == "__main__":
    mcp.run(transport="stdio")
"""

SERVER_FILE.write_text(server_code, encoding="utf-8")
print(f"Serveur MCP créé : {SERVER_FILE.resolve()}")

Serveur MCP créé : /home/thomas/Desktop/CODE/CEPE/cours_agents_td/mcp_tp_server.py


## 3. Où est lancé le serveur MCP ?

Ici, le serveur MCP est lancé par le notebook. Cette ligne ne lance pas encore le serveur :

```python
server_params = StdioServerParameters(...)
```

Elle explique seulement **comment** le lancer. Le serveur est lancé réellement quand on entre dans :

```python
async with stdio_client(server_params) as (read, write):
```

Cela crée un sous-processus équivalent à :
```bash
python mcp_tp_server.py
```

Le serveur communique ensuite avec le notebook via `stdin/stdout`. C'est le mode MCP `stdio`. Donc pour ce TP :

> Vous lancez le notebook, pas le serveur à la main.

Si vous testez `python mcp_tp_server.py` dans un terminal, c'est normal que le programme semble attendre : il attend qu'un client MCP lui parle via stdio.

In [8]:
server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(SERVER_FILE)]
)

server_params

StdioServerParameters(command='/home/thomas/Desktop/CODE/CEPE/cours_agents_td/.venv/bin/python', args=['mcp_tp_server.py'], env=None, cwd=None, encoding='utf-8', encoding_error_handler='strict')

## 4. Fonctions utilitaires MCP

Ces fonctions permettent de :

- récupérer le schema d'un tool MCP ;
- convertir un résultat MCP en objet Python ;
- lister les tools exposés par le serveur ;
- appeler un tool MCP.

`async` veut dire : “cette fonction ou ce bloc contient des attentes”
`await` signifie veut dire : “attends le résultat”, par exemple la ligne de code
```python
tools_result = await session.list_tools()
```
veut dire : appelle session.list_tools(), mais comme cette fonction parle à un serveur externe, elle ne répond pas instantanément. Attends que le résultat arrive.

In [9]:
def get_tool_schema(tool):
    """Récupère le schéma d'entrée d'un tool MCP, avec compatibilité selon versions du SDK."""
    return getattr(tool, "inputSchema", None) or getattr(tool, "input_schema", None) or {}


def mcp_result_to_python(result):
    """Convertit un résultat MCP en objet Python quand c'est possible."""
    structured = getattr(result, "structuredContent", None)
    if structured is not None:
        return structured

    texts = []
    for item in getattr(result, "content", []):
        text = getattr(item, "text", None)
        if text is not None:
            texts.append(text)

    if texts:
        text = "\n".join(texts)
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return text

    return str(result)


async def list_mcp_tools():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_result = await session.list_tools()
            return tools_result.tools


async def call_mcp_tool(tool_name: str, arguments: dict):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments=arguments)
            return mcp_result_to_python(result)

# Partie 1 — MCP sans LLM

OpenAI n'intervient pas encore.

Objectif : voir que MCP fonctionne déjà comme un protocole client ↔ serveur.

## 5. Découvrir les tools exposés par le serveur

Le client MCP demande au serveur :

> Quels tools exposes-tu ?

C'est la découverte dynamique des capabilities.

In [10]:
mcp_tools = await list_mcp_tools()

for tool in mcp_tools:
    print("=" * 80)
    print("Nom :", tool.name)
    print("Description :", tool.description)
    print("Schéma d'entrée :")
    pprint(get_tool_schema(tool))

Nom : calculate_cart_total
Description : 
    Calculate the final price of a shopping cart.

    Rules:
    - Sum item prices.
    - Apply discount.
    - Apply tax after discount.
    - Add shipping unless total with tax before shipping is strictly greater than the free shipping threshold.
    - Round values to two decimals.
    
Schéma d'entrée :
{'properties': {'discount_percent': {'title': 'Discount Percent',
                                     'type': 'number'},
                'free_shipping_threshold': {'title': 'Free Shipping Threshold',
                                            'type': 'number'},
                'item_prices': {'items': {'type': 'number'},
                                'title': 'Item Prices',
                                'type': 'array'},
                'shipping_fee': {'title': 'Shipping Fee', 'type': 'number'},
                'tax_percent': {'title': 'Tax Percent', 'type': 'number'}},
 'required': ['item_prices',
              'discount_percent',
 

## 6. Appeler un tool MCP directement

On appelle maintenant `calculate_cart_total`, sans LLM.

In [11]:
cart_result = await call_mcp_tool(
    "calculate_cart_total",
    {
        "item_prices": [79.90, 39.50, 249.99],
        "discount_percent": 12,
        "tax_percent": 20,
        "shipping_fee": 8.90,
        "free_shipping_threshold": 350
    }
)

pprint(cart_result)

{'discounted_total': 325.06,
 'final_total': 390.08,
 'shipping_applied': 0,
 'subtotal': 369.39,
 'total_with_tax_before_shipping': 390.08}


# Partie 2 — MCP + OpenAI

Maintenant, OpenAI choisit le tool, mais c'est toujours le serveur MCP qui exécute le vrai code.

## 7. Convertir les tools MCP en tools OpenAI

OpenAI attend une liste de tools au format function calling.

Un tool MCP contient déjà :

- un nom ;
- une description ;
- un schema d'entrée.

On peut donc traduire les tools MCP vers le format attendu par OpenAI.

In [12]:
def mcp_tool_to_openai_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "",
            "parameters": get_tool_schema(tool),
            "strict": False
        }
    }


openai_tools = [mcp_tool_to_openai_tool(tool) for tool in mcp_tools]

pprint(openai_tools)

[{'function': {'description': '\n'
                              '    Calculate the final price of a shopping '
                              'cart.\n'
                              '\n'
                              '    Rules:\n'
                              '    - Sum item prices.\n'
                              '    - Apply discount.\n'
                              '    - Apply tax after discount.\n'
                              '    - Add shipping unless total with tax before '
                              'shipping is strictly greater than the free '
                              'shipping threshold.\n'
                              '    - Round values to two decimals.\n'
                              '    ',
               'name': 'calculate_cart_total',
               'parameters': {'properties': {'discount_percent': {'title': 'Discount '
                                                                           'Percent',
                                                  

Question :

> Est-ce qu'OpenAI découvre directement les tools MCP ?

Réponse attendue : non.  
Le notebook découvre les tools via MCP, puis les convertit dans le format OpenAI.

## 8. Prompt de test

Le modèle doit comprendre la demande, choisir le bon tool et extraire les arguments.

Le notebook exécutera ensuite le tool via MCP.

In [13]:
SHOP_PROMPT = '''
J’ai acheté 3 articles :
- un clavier à 79,90 €
- une souris à 39,50 €
- un écran à 249,99 €

J’ai un code promo de 12 %, puis je dois ajouter une TVA de 20 %.
Les frais de livraison sont de 8,90 €, mais ils sont offerts si le total TTC avant livraison dépasse 350 €.

Quel est le montant final à payer ?
'''

## 9. Premier appel OpenAI : demander un tool_call

OpenAI ne va pas exécuter le tool. Il va seulement demander un appel de tool.

In [10]:
messages = [
    {
        "role": "user",
        "content": SHOP_PROMPT
    }
]

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
    tools=openai_tools
)

assistant_message = response.choices[0].message

pprint(assistant_message.model_dump())

{'annotations': [],
 'audio': None,
 'content': None,
 'function_call': None,
 'refusal': None,
 'role': 'assistant',
 'tool_calls': [{'function': {'arguments': '{"item_prices":[79.9,39.5,249.99],"discount_percent":12,"tax_percent":20,"shipping_fee":8.9,"free_shipping_threshold":350}',
                              'name': 'calculate_cart_total'},
                 'id': 'call_jPaNJQdghv3NuFoFYHEypaM5',
                 'type': 'function'}]}


## 10. Exécuter via MCP le tool demandé par OpenAI

In [14]:
if not assistant_message.tool_calls:
    raise RuntimeError("Le modèle n'a pas demandé de tool_call. Relancez la cellule ou forcez tool_choice si besoin.")

tool_call = assistant_message.tool_calls[0]

tool_name = tool_call.function.name
tool_arguments = json.loads(tool_call.function.arguments)

print("Tool demandé par OpenAI :", tool_name)
print("Arguments extraits par OpenAI :")
pprint(tool_arguments)

mcp_output = await call_mcp_tool(tool_name, tool_arguments)

print("\nRésultat renvoyé par le serveur MCP :")
pprint(mcp_output)

NameError: name 'assistant_message' is not defined

Questions :

1. Qui a choisi le tool ?
2. Qui a extrait les arguments ?
3. Qui a exécuté le code réel ?
4. Qui a produit le résultat numérique ?

## 11. Renvoyer le résultat MCP à OpenAI

Maintenant que le tool a été exécuté, on renvoie le résultat à OpenAI.

Le modèle peut alors produire une réponse finale en langage naturel.

In [15]:
def assistant_message_to_dict(message):
    data = {
        "role": "assistant",
        "content": message.content
    }

    if message.tool_calls:
        data["tool_calls"] = [tool_call.model_dump() for tool_call in message.tool_calls]

    return data


messages.append(assistant_message_to_dict(assistant_message))

messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(mcp_output, ensure_ascii=False)
})

final_response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=messages,
    tools=openai_tools
)

print(final_response.choices[0].message.content)

NameError: name 'messages' is not defined

## 12. Mini-boucle agentique OpenAI + MCP

On regroupe tout dans une fonction.

La fonction :

1. se connecte au serveur MCP ;
2. découvre les tools ;
3. convertit les tools MCP en tools OpenAI ;
4. appelle OpenAI ;
5. exécute les tool calls via MCP ;
6. renvoie les résultats à OpenAI ;
7. s'arrête quand OpenAI produit une réponse finale.

In [ ]:
async def ask_openai_using_mcp(prompt: str, model: str = 'gpt-4o-mini'):
    """Boucle agentique simple : OpenAI choisit les tools, MCP les exécute.

    Étapes pédagogiques :
    1. Connexion au serveur MCP (client ↔ serveur).
    2. Découverte dynamique des tools exposés par le serveur MCP.
    3. Appel du modèle OpenAI en lui donnant ces tools (au format function calling).
    4. Pour chaque `tool_call` demandé par le modèle, exécution réelle côté MCP.
    5. Renvoi du résultat du tool au modèle, jusqu'à obtenir une réponse finale.
    """

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            openai_tools = [mcp_tool_to_openai_tool(tool) for tool in tools_result.tools]

            messages = [
                {
                    "role": "user",
                    "content": prompt
                }
            ]

            step = 1

            while True:
                print(f"\n=== Tour {step} : appel OpenAI ===")
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    tools=openai_tools
                )

                assistant_message = response.choices[0].message
                messages.append(assistant_message_to_dict(assistant_message))

                # Cas 1 : le modèle ne demande plus de tool_call → réponse finale.
                if not assistant_message.tool_calls:
                    print("Aucun tool supplémentaire demandé : réponse finale générée par le modèle.")
                    return assistant_message.content

                # Cas 2 : le modèle demande un ou plusieurs tool_calls → on les exécute via MCP.
                for tool_call in assistant_message.tool_calls:
                    tool_name = tool_call.function.name
                    arguments = json.loads(tool_call.function.arguments)

                    print(f"Tool demandé : {tool_name}")
                    print("Arguments extraits par le modèle :")
                    pprint(arguments)

                    mcp_result = await session.call_tool(tool_name, arguments=arguments)
                    mcp_output = mcp_result_to_python(mcp_result)

                    print("Résultat renvoyé par le serveur MCP :")
                    pprint(mcp_output)
                    print("-" * 80)

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": json.dumps(mcp_output, ensure_ascii=False)
                    })

                step += 1

## 13. Tester la boucle complète

In [17]:
answer = await ask_openai_using_mcp(SHOP_PROMPT)

print("\nRéponse finale :")
print(answer)


=== Tour 1 : appel OpenAI ===
Tool demandé : calculate_cart_total
Arguments extraits par le modèle :
{'discount_percent': 12,
 'free_shipping_threshold': 350,
 'item_prices': [79.9, 39.5, 249.99],
 'shipping_fee': 8.9,
 'tax_percent': 20}
Résultat renvoyé par le serveur MCP :
{'discounted_total': 325.06,
 'final_total': 390.08,
 'shipping_applied': 0,
 'subtotal': 369.39,
 'total_with_tax_before_shipping': 390.08}
--------------------------------------------------------------------------------

=== Tour 2 : appel OpenAI ===
Aucun tool supplémentaire demandé : réponse finale générée par le modèle.

Réponse finale :
Le montant final à payer est de 390,08 €. Les frais de livraison sont offerts car le total TTC avant livraison dépasse 350 €.


# Exercice

Ajoutez un nouveau tool au serveur MCP.

Exemple : `calculate_rental_price`.

Règles métier :

- une voiture est louée entre une date de début et une date de fin ;
- tout jour commencé est facturé ;
- tarif de base par jour ;
- assurance par jour ;
- supplément jeune conducteur si l'âge du conducteur est inférieur à 25 ans ;
- frais de dossier fixes ;
- remise de 10 % si la location dure au moins 5 jours commencés.

In [ ]:
import datetime


def calculate_rental_price(start_date: str, end_date: str, driver_age: int) -> dict:
    """
    Calculate the rental price for a car rental.

    Rules:
    - The rental price is calculated based on the start and end dates of the rental, the driver's age,
    and the type of car.

    - une voiture est louée entre une date de début et une date de fin ;
    - tout jour commencé est facturé ;
    - tarif de base par jour ;
    - assurance par jour ;
    - supplément jeune conducteur si l'âge du conducteur est inférieur à 25 ans ;
    - frais de dossier fixes ;
    - remise de 10 % si la location dure au moins 5 jours commencés.
    """

    if start_date > end_date:
        return {
            "error": "start_date_after_end_date",
            "message": "The start date cannot be after the end date."
        }

    # Conversion des dates en datetime
    end_date = datetime.strptime(end_date, "%Y-%m-%d")
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    # Calcul du nombre de jours de location
    days = (end_date - start_date).days
    # Tarif de base par jour
    base_price = days * 50
    # Remise de 10 % si la location dure au moins 5 jours commencés.
    if days >= 5:
        base_price *= 0.9
    
    if driver_age < 25:
        base_price += days * 10
    
    base_price += 100

    return {
        "rental_price": base_price
    }


In [24]:
SERVER_FILE = Path("mcp_exercice_server.py")

server_code = """
from mcp.server.fastmcp import FastMCP
import datetime

mcp = FastMCP("tp-mcp-demo")

@mcp.tool()
def calculate_rental_price(start_date: str, end_date: str, driver_age: int) -> dict:
    \"""
    Calculate the rental price for a car rental.

    Rules:
    - The rental price is calculated based on the start and end dates of the rental, the driver's age,
    and the type of car.

    - une voiture est louée entre une date de début et une date de fin ;
    - tout jour commencé est facturé ;
    - tarif de base par jour ;
    - assurance par jour ;
    - supplément jeune conducteur si l'âge du conducteur est inférieur à 25 ans ;
    - frais de dossier fixes ;
    - remise de 10 % si la location dure au moins 5 jours commencés.
    \"""
    if start_date > end_date:
        return {
            "error": "start_date_after_end_date",
            "message": "The start date cannot be after the end date."
        }
    # Conversion des dates en datetime
    end_date = datetime.strptime(end_date, "%Y-%m-%d")
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    # Calcul du nombre de jours de location
    days = (end_date - start_date).days
    # Tarif de base par jour
    base_price = days * 50
    # Remise de 10 % si la location dure au moins 5 jours commencés.
    if days >= 5:
        base_price *= 0.9
    
    if driver_age < 25:
        base_price += days * 10
    
    base_price += 100

    return {
        "rental_price": base_price
    }

if __name__ == "__main__":
    mcp.run(transport="stdio")
"""

SERVER_FILE.write_text(server_code, encoding="utf-8")
print(f"Serveur MCP créé : {SERVER_FILE.resolve()}")

Serveur MCP créé : /home/thomas/Desktop/CODE/CEPE/cours_agents_td/mcp_exercice_server.py


Ok on a cree un nouveau fichier 'mcp_exerice_server'. Mais on doit
- lancer le serveur avec le nouveau mcp
- ecrire la boucle agentique avec l'appel au serveur : on peut utiliser directement la fonction `ask_openai_using_mcp`

In [26]:
server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(SERVER_FILE)]
)

In [27]:
async def list_mcp_tools():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_result = await session.list_tools()
            return tools_result.tools


async def call_mcp_tool(tool_name: str, arguments: dict):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments=arguments)
            return mcp_result_to_python(result)

In [28]:
async def ask_openai_using_mcp(prompt: str, model: str = 'gpt-4o-mini'):
    """Boucle agentique simple : OpenAI choisit les tools, MCP les exécute.

    Étapes pédagogiques :
    1. Connexion au serveur MCP (client ↔ serveur).
    2. Découverte dynamique des tools exposés par le serveur MCP.
    3. Appel du modèle OpenAI en lui donnant ces tools (au format function calling).
    4. Pour chaque `tool_call` demandé par le modèle, exécution réelle côté MCP.
    5. Renvoi du résultat du tool au modèle, jusqu'à obtenir une réponse finale.
    """

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            openai_tools = [mcp_tool_to_openai_tool(tool) for tool in tools_result.tools]

            messages = [
                {
                    "role": "user",
                    "content": prompt
                }
            ]

            step = 1

            while True:
                print(f"\n=== Tour {step} : appel OpenAI ===")
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    tools=openai_tools
                )

                assistant_message = response.choices[0].message
                messages.append(assistant_message_to_dict(assistant_message))

                # Cas 1 : le modèle ne demande plus de tool_call → réponse finale.
                if not assistant_message.tool_calls:
                    print("Aucun tool supplémentaire demandé : réponse finale générée par le modèle.")
                    return assistant_message.content

                # Cas 2 : le modèle demande un ou plusieurs tool_calls → on les exécute via MCP.
                for tool_call in assistant_message.tool_calls:
                    tool_name = tool_call.function.name
                    arguments = json.loads(tool_call.function.arguments)

                    print(f"Tool demandé : {tool_name}")
                    print("Arguments extraits par le modèle :")
                    pprint(arguments)

                    mcp_result = await session.call_tool(tool_name, arguments=arguments)
                    mcp_output = mcp_result_to_python(mcp_result)

                    print("Résultat renvoyé par le serveur MCP :")
                    pprint(mcp_output)
                    print("-" * 80)

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": json.dumps(mcp_output, ensure_ascii=False)
                    })

                step += 1

In [29]:

USER_PROMPT = 'Je viens de louer la voiture entre le mardi 6 Mai et le vendredi 18 Mai. Combien ca coute ?'
answer = await ask_openai_using_mcp(USER_PROMPT)

print("\nRéponse finale :")
print(answer)


=== Tour 1 : appel OpenAI ===
Tool demandé : calculate_rental_price
Arguments extraits par le modèle :
{'driver_age': 30, 'end_date': '2024-05-18', 'start_date': '2024-05-06'}
Résultat renvoyé par le serveur MCP :
("Error executing tool calculate_rental_price: module 'datetime' has no "
 "attribute 'strptime'")
--------------------------------------------------------------------------------
Tool demandé : calculate_rental_price
Arguments extraits par le modèle :
{'driver_age': 22, 'end_date': '2024-05-18', 'start_date': '2024-05-06'}
Résultat renvoyé par le serveur MCP :
("Error executing tool calculate_rental_price: module 'datetime' has no "
 "attribute 'strptime'")
--------------------------------------------------------------------------------

=== Tour 2 : appel OpenAI ===
Aucun tool supplémentaire demandé : réponse finale générée par le modèle.

Réponse finale :
Il semble y avoir un problème technique pour le moment. Pour vous aider à calculer le prix de la location de la voitur

# Conclusion

Dans ce TP :

```text
Notebook Jupyter = host
Code Python dans le notebook = client MCP
mcp_tp_server.py = serveur MCP
calculate_cart_total / divide = tools MCP
stdio = transport
OpenAI = modèle qui demande un tool_call
```

Phrase à retenir :

> OpenAI choisit un tool, mais n'exécute pas le tool.  
> Le notebook reçoit le tool_call et appelle le serveur MCP.  
> MCP standardise l'accès aux outils ; le function calling standardise la demande d'utilisation d'un outil par le modèle.